# Wildfire Prediction CNN - PyTorch Version

This notebook implements a CNN for wildfire prediction using PyTorch, converted from the TensorFlow version.

In [ ]:
# Environment setup
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Importing Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import cv2
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [ ]:
# Check available device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 2. Gathering Data

In [ ]:
# Defining paths of train, validation and test data
# For FASDD_RS dataset structure (if using the dataset from config.py):
DATASET_ROOT = "/mnt/storage/Dataset/FASDD_RS"
train_path = os.path.join(DATASET_ROOT, "train")
valid_path = os.path.join(DATASET_ROOT, "val")
test_path = os.path.join(DATASET_ROOT, "test")

In [ ]:
# Hyperparameters
IMAGE_SIZE = 350
N_CLASSES = 2
BATCH_SIZE = 256
EPOCHS = 50
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-5

print(f"Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Number of Classes: {N_CLASSES}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")

In [ ]:
# Data augmentation and normalization
# For training data
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# For validation and test data (no augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Load datasets using ImageFolder
train_dataset = ImageFolder(root=train_path, transform=train_transform)
valid_dataset = ImageFolder(root=valid_path, transform=val_test_transform)
test_dataset = ImageFolder(root=test_path, transform=val_test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                         num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=4, pin_memory=True)

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(valid_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nClass names: {train_dataset.classes}")
print(f"Class to index mapping: {train_dataset.class_to_idx}")

## 3. Building the Model

In [ ]:
class WildfireCNN(nn.Module):
    """CNN model for wildfire prediction - PyTorch version"""
    
    def __init__(self, num_classes=2, weight_decay=1e-3):
        super(WildfireCNN, self).__init__()
        
        # First convolutional block
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=2, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional block
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=2, padding=0)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Third convolutional block (with regularization)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=2, padding=0)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout layers
        self.dropout1 = nn.Dropout(p=0.4)
        self.dropout2 = nn.Dropout(p=0.5)
        
        # Calculate the flattened size
        # After 3 conv+pool layers with kernel_size=2 and no padding:
        # 350 -> 349 -> 174 -> 173 -> 86 -> 85 -> 42
        self.flatten_size = 32 * 42 * 42
        
        # Fully connected layers
        self.fc1 = nn.Linear(self.flatten_size, 300)
        self.fc2 = nn.Linear(300, num_classes)
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights using He initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # First conv block
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        
        # Second conv block
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        
        # Third conv block
        x = F.relu(self.conv3(x))
        x = self.pool3(x)
        
        # Dropout and flatten
        x = self.dropout1(x)
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        
        return x

In [ ]:
# Create model and move to device
model = WildfireCNN(num_classes=N_CLASSES)
model = model.to(device)

# Print model summary
print(model)
print("\n" + "="*60)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 4. Training the Model

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler (optional, similar to early stopping behavior)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print("Optimizer: Adam")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Weight Decay: {WEIGHT_DECAY}")

In [ ]:
def calculate_auc(outputs, labels):
    """Calculate AUC metric (simplified binary version)"""
    from sklearn.metrics import roc_auc_score
    
    # Apply softmax to get probabilities
    probs = F.softmax(outputs, dim=1).detach().cpu().numpy()
    labels_np = labels.cpu().numpy()
    
    try:
        if N_CLASSES == 2:
            # Binary classification
            auc = roc_auc_score(labels_np, probs[:, 1])
        else:
            # Multi-class (one-vs-rest)
            auc = roc_auc_score(labels_np, probs, multi_class='ovr', average='macro')
        return auc
    except:
        return 0.0


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_outputs = []
    all_labels = []
    
    pbar = tqdm(dataloader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_outputs.append(outputs)
        all_labels.append(labels)
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    # Calculate AUC
    all_outputs = torch.cat(all_outputs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    epoch_auc = calculate_auc(all_outputs, all_labels)
    
    return epoch_loss, epoch_acc, epoch_auc


def validate(model, dataloader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_outputs = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_outputs.append(outputs)
            all_labels.append(labels)
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    # Calculate AUC
    all_outputs = torch.cat(all_outputs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    epoch_auc = calculate_auc(all_outputs, all_labels)
    
    return epoch_loss, epoch_acc, epoch_auc

In [ ]:
# Training loop with early stopping
history = {
    'train_loss': [],
    'train_acc': [],
    'train_auc': [],
    'val_loss': [],
    'val_acc': [],
    'val_auc': []
}

best_val_loss = float('inf')
patience = 10
patience_counter = 0
best_model_path = 'first_model_pytorch.pth'

print("Starting training...\n")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 60)
    
    # Train
    train_loss, train_acc, train_auc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_acc, val_auc = validate(
        model, valid_loader, criterion, device
    )
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    # Print epoch summary
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Train AUC: {train_auc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val AUC: {val_auc:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, best_model_path)
        print(f"✓ Model saved with val_loss: {val_loss:.4f}")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered after epoch {epoch+1}")
        break

print("\n" + "="*60)
print("Training completed!")
print(f"Best validation loss: {best_val_loss:.4f}")

## 5. Visualizing Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy plot
axes[0].plot(history['train_acc'], label='Train Accuracy')
axes[0].plot(history['val_acc'], label='Validation Accuracy')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Model Accuracy')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history['train_loss'], label='Train Loss')
axes[1].plot(history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# AUC plot
axes[2].plot(history['train_auc'], label='Train AUC')
axes[2].plot(history['val_auc'], label='Validation AUC')
axes[2].set_xlabel('Epochs')
axes[2].set_ylabel('AUC')
axes[2].set_title('Model AUC')
axes[2].legend(loc='lower right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Testing the Model

In [ ]:
# Load the best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Best validation accuracy: {checkpoint['val_acc']:.2f}%")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_acc, test_auc = validate(model, test_loader, criterion, device)

print("\n" + "="*60)
print("Test Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test AUC: {test_auc:.4f}")
print("="*60)

In [ ]:
# Detailed evaluation with confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=test_dataset.classes, 
            yticklabels=test_dataset.classes)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, 
                          target_names=test_dataset.classes))

## 7. Sample Predictions

In [ ]:
# Visualize some predictions
def imshow(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Denormalize and show image"""
    img = img.cpu().numpy().transpose((1, 2, 0))
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

# Get a batch of test images
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Make predictions
model.eval()
with torch.no_grad():
    images_gpu = images.to(device)
    outputs = model(images_gpu)
    probs = F.softmax(outputs, dim=1)
    _, predicted = outputs.max(1)

# Display first 8 images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx in range(8):
    img = imshow(images[idx])
    axes[idx].imshow(img)
    
    true_label = test_dataset.classes[labels[idx]]
    pred_label = test_dataset.classes[predicted[idx]]
    confidence = probs[idx][predicted[idx]].item() * 100
    
    color = 'green' if labels[idx] == predicted[idx] else 'red'
    axes[idx].set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
                       color=color, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 8. Save Final Model

In [ ]:
# Save complete model for inference
final_model_path = 'wildfire_cnn_pytorch_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': test_dataset.classes,
    'image_size': IMAGE_SIZE,
    'num_classes': N_CLASSES,
    'test_acc': test_acc,
    'test_auc': test_auc
}, final_model_path)

print(f"Final model saved to: {final_model_path}")